# Solution 01: GLiNER NER Demo

In [ ]:
import json, os
import gradio as gr
from gliner import GLiNER

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "sample_texts.json")) as f:
    SAMPLE_TEXTS = json.load(f)

ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]

ENTITY_COLORS = {
    "malware": "#ff6b6b",
    "vulnerability": "#ffa94d",
    "attack_technique": "#74c0fc",
    "affected_software": "#69db7c",
    "threat_actor": "#da77f2",
}

print(f"✓ Loaded {len(SAMPLE_TEXTS)} sample texts")

In [ ]:
model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ GLiNER loaded")

## Part 1: Implement `run_ner`

In [ ]:
def run_ner(text: str, entity_types: list, threshold: float) -> list:
    if not text.strip() or not entity_types:
        return [(text, None)]

    entities = model.predict_entities(text, entity_types, threshold=threshold)
    entities = sorted(entities, key=lambda e: e["start"])

    result = []
    cursor = 0
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        if start > cursor:
            result.append((text[cursor:start], None))
        result.append((text[start:end], label))
        cursor = end
    if cursor < len(text):
        result.append((text[cursor:], None))
    return result if result else [(text, None)]


result = run_ner(SAMPLE_TEXTS[0]['text'], ENTITY_TYPES, threshold=0.4)
assert isinstance(result, list)
assert all(isinstance(item, tuple) and len(item) == 2 for item in result)
labels_found = [label for _, label in result if label is not None]
assert len(labels_found) >= 1
assert all(label in ENTITY_TYPES for label in labels_found)
full_text = "".join(chunk for chunk, _ in result)
assert full_text == SAMPLE_TEXTS[0]['text']
print(f"✓ run_ner works: {[(label, chunk[:25]) for chunk, label in result if label]}")

## Part 2: Build `gr.Blocks` Demo

In [ ]:
with gr.Blocks(title="NER Demo") as demo:
    gr.Markdown("# GLiNER — Named Entity Recognition\nZero-shot NER for cybersecurity texts.")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Input text",
                lines=5,
                placeholder="Paste cybersecurity report here..."
            )
            entity_types = gr.CheckboxGroup(
                choices=ENTITY_TYPES,
                value=ENTITY_TYPES,
                label="Entity types"
            )
            threshold = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Confidence threshold")
            run_btn = gr.Button("Extract entities", variant="primary")

        with gr.Column(scale=2):
            ner_output = gr.HighlightedText(
                label="Entities",
                color_map=ENTITY_COLORS,
                show_legend=True,
            )

    run_btn.click(run_ner, inputs=[text_input, entity_types, threshold], outputs=ner_output)

assert 'demo' in dir()
assert hasattr(demo, 'launch')
print("✓ demo defined")
demo.launch()

## Part 3: Add `gr.Examples`

In [ ]:
with gr.Blocks(title="NER Demo") as demo_with_examples:
    gr.Markdown("# GLiNER — Named Entity Recognition")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Input text",
                lines=5,
                placeholder="Paste cybersecurity report here..."
            )
            entity_types = gr.CheckboxGroup(
                choices=ENTITY_TYPES,
                value=ENTITY_TYPES,
                label="Entity types"
            )
            threshold = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Confidence threshold")
            run_btn = gr.Button("Extract entities", variant="primary")

        with gr.Column(scale=2):
            ner_output = gr.HighlightedText(
                label="Entities",
                color_map=ENTITY_COLORS,
                show_legend=True,
            )

    gr.Examples(
        examples=[[t['text']] for t in SAMPLE_TEXTS[:4]],
        inputs=text_input,
        label="Example reports"
    )

    run_btn.click(run_ner, inputs=[text_input, entity_types, threshold], outputs=ner_output)

assert 'demo_with_examples' in dir()
print("✓ demo_with_examples defined")
demo_with_examples.launch()